# पाठ ०५ - एजेन्टिक RAG


## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## एजेन्टिक RAG के हो?

परम्परागत RAG एक निश्चित पाइपलाइन अनुसरण गर्दछ: कागजातहरू प्राप्त गर्नुहोस्, त्यसपछि उत्तर सिर्जना गर्नुहोस्। **एजेन्टिक RAG** ले अगाडि बढेर एजेन्टलाई स्वतन्त्रता दिन्छ **कहिले** र **कसरी** जानकारी प्राप्त गर्ने निर्णय गर्न।

एजेन्टिक RAG सँग, एजेन्टले सक्छ:
- प्रश्नको उत्तर दिनुअघि प्राप्ति आवश्यक छ कि छैन **निर्णय लिन**
- कुन डाटा स्रोत वा उपकरणलाई सोध्ने **छान्नुहोस्**
- प्राप्त नतिजाहरू **मूल्यांकन गर्नुहोस्** र पहिलो प्रयास पर्याप्त नभएमा थप प्राप्तिहरू गर्नुहोस्
- विभिन्न प्राप्ति चरणहरूको जानकारीलाई एक सुसंगत उत्तरमा **संयोजन गर्नुहोस्**

यसले एजेन्टलाई स्थिर retrieve-then-generate पाइपलाइनको तुलनामा बढी लचकदार र सटीक बनाउँछ।


## खोज उपकरण सिर्जना गर्दै

Agentic RAG मा, बाह्य डाटा स्रोतहरूलाई **उपकरणहरू** को रूपमा लिपेटिन्छ जुन एजेन्टले आवश्यक पर्दा बोलाउन सक्छ। यसले एजेन्टलाई पुनःप्राप्तिलाई एउटा अनिवार्य कदमको सट्टा लिन सक्ने अर्को कार्यको रूपमा व्यवहार गर्न दिन्छ।

तल हामीले यात्रा ज्ञान आधार परिभाषित गरेका छौं र यसलाई एजेन्टले गन्तव्य जानकारी हेर्न सक्ने उपकरणको रूपमा प्रस्तुत गरेका छौं।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजेन्ट निर्माण गर्दै

अब हामी एउटा एजेन्ट बनाउँछौं जुन **सधैं उत्तर दिनु अघि जानकारी प्राप्त गर्ने** निर्देशन पाएको हुन्छ। एजेन्टले `search_travel_knowledge` उपकरण प्रयोग गर्छ जसले यसको उत्तरहरू आफ्नै प्रशिक्षण डेटा भन्दा ज्ञान आधारमा आधारित हुन्छ।


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावृत्तिक पुन:प्राप्ति — मेकर-चेकर ढाँचा

एजेन्टिक RAG को एक प्रमुख फाइदा हो **पुनरावृत्तिक पुन:प्राप्ति**। एजेन्टले आफ्नो प्रारम्भिक पत्ता लगाउने कुरालाई पुष्टि गर्न, सुधार गर्न, वा विस्तार गर्न धेरै पटक खोज गर्न सक्छ — "मेकर-चेकर" कार्यप्रवाह जस्तै:

1. **मेकर चरण**: एजेन्टले प्रारम्भिक जानकारी पुन:प्राप्ति गर्छ र जवाफको मसौदा तयार गर्छ।
२. **चेकर चरण**: एजेन्टले विवरणहरू पुष्टि गर्न वा खाली ठाउँहरू भर्न अतिरिक्त पुन:प्राप्तिहरू गर्छ।

तल, एजेन्टलाई एउटा प्रश्न सोधिएको छ जसले धेरै गन्तव्यहरूको तुलना गर्नुपर्छ, जसले यसलाई धेरै पटक खोज गर्न प्रेरित गर्छ।


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

यस पाठमा तपाईंले Microsoft Agent Framework प्रयोग गरेर **Agentic RAG** प्रणाली कसरी बनाउने सिक्नुभयो:

- **Agentic RAG** ले एजेन्टहरूलाई स्वायत्त रूपमा कहिले जानकारी पुनः प्राप्त गर्ने निर्णय गर्न दिन्छ, जसले पुनः प्राप्तिलाई स्थिर नभई गतिशील बनाउँछ।
- **साधनहरूलाई डाटा स्रोतका रूपमा**: बाह्य ज्ञान आधारहरू (जस्तै Azure AI Search) लाई एजेन्टले कल गर्न सक्ने साधनहरूका रूपमा समेटिन्छ।
- **क्रमागत पुनः प्राप्ति**: निर्माता-जाँचकर्ता ढाँचाले एजेन्टलाई धेरै पुनः प्राप्ति चरणहरू — खोज्दै, प्रमाणीकरण गर्दै र सुधार गर्दै — अन्तिम उत्तर निकाल्न सक्षम बनाउँछ।

उत्पादनमा, ठूलो मात्रामा यात्रा कागजात पुनः प्राप्तिका लागि तपाईंले इन-मेमोरी `TRAVEL_KNOWLEDGE_BASE` लाई वास्तविक Azure AI Search इन्डेक्सले प्रतिस्थापन गर्नुहुनेछ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
